In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

# Configurações iniciais
n_tickets = 5000
np.random.seed(42)

# Gerando datas de abertura nos últimos 6 meses
start_date = datetime.now() - timedelta(days=180)
abertura = [start_date + timedelta(minutes=random.randint(0, 180 * 24 * 60)) for _ in range(n_tickets)]

categorias = ['Hardware', 'Software', 'Redes', 'Acesso/Credenciais', 'Dúvida do Usuário']
prioridades = ['Baixa', 'Média', 'Alta', 'Crítica']
status = ['Fechado', 'Fechado', 'Fechado', 'Fechado', 'Em Andamento'] # Maioria fechado

df_hd = pd.DataFrame({
    'id_chamado': range(1001, 1001 + n_tickets),
    'data_abertura': abertura,
    'categoria': np.random.choice(categorias, n_tickets, p=[0.2, 0.3, 0.15, 0.25, 0.1]),
    'prioridade': np.random.choice(prioridades, n_tickets, p=[0.4, 0.4, 0.15, 0.05]),
    'status': np.random.choice(status, n_tickets)
})

# Lógica de tempo de resolução baseada na prioridade
def calcular_fechamento(row):
    if row['status'] == 'Em Andamento':
        return pd.NaT
    
    # Horas baseadas na prioridade
    if row['prioridade'] == 'Crítica': horas = np.random.uniform(0.5, 4)
    elif row['prioridade'] == 'Alta': horas = np.random.uniform(2, 12)
    elif row['prioridade'] == 'Média': horas = np.random.uniform(12, 48)
    else: horas = np.random.uniform(24, 120)
    
    return row['data_abertura'] + timedelta(hours=horas)

df_hd['data_fechamento'] = df_hd.apply(calcular_fechamento, axis=1)

# Avaliação de satisfação (1 a 5)
df_hd['satisfacao_usuario'] = np.where(
    df_hd['status'] == 'Fechado',
    np.random.choice([1, 2, 3, 4, 5], n_tickets, p=[0.05, 0.1, 0.15, 0.4, 0.3]),
    np.nan
)

# Conexão segura e envio para o banco
load_dotenv()
engine = create_engine(f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASS')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}")

df_hd.to_sql('help_desk_tickets', engine, if_exists='replace', index=False)
print("Tabela 'help_desk_tickets' criada com sucesso no PostgreSQL!")

Tabela 'help_desk_tickets' criada com sucesso no PostgreSQL!


In [1]:
import os
from dotenv import load_dotenv

# Força o recarregamento do arquivo .env
load_dotenv(override=True)

print("--- DIAGNÓSTICO DE CONEXÃO ---")
print(f"Usuário: {os.getenv('DB_USER')}")
print(f"Host: {os.getenv('DB_HOST')}")
print(f"Porta: {os.getenv('DB_PORT')}")
print(f"Banco Alvo: {os.getenv('DB_NAME')}")
print("------------------------------")

--- DIAGNÓSTICO DE CONEXÃO ---
Usuário: postgres
Host: localhost
Porta: 5432
Banco Alvo: logistica_supply
------------------------------
